# HFT Toxicity Circuit Breaker: Interactive Demo

1. **Free tier:** replay live FollowSM REST snapshots through the circuit breaker to watch it trip on VPIN percentile > 0.90, 1% book toxicity, and depth-imbalance spikes, and measure its in-process evaluation latency.
2. **Enterprise:** run the real tick-level breaker on `/ws/v1/toxicity` + `/ws/v1/confluence` for one minute.

Enterprise keys: https://follow-sm.com/pricing

In [ ]:
!git clone -q https://github.com/Follow-SM/hft-toxicity-circuit-breaker.git
%cd hft-toxicity-circuit-breaker
!pip install -q -r requirements.txt

In [ ]:
import asyncio
import logging

from followsm_sdk import FollowSMClient, RateLimitExceededException

from circuit_breaker.breaker import CircuitBreaker
from circuit_breaker.callbacks import log_event
from circuit_breaker.config import BreakerConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s", force=True)
client = FollowSMClient()  # free tier: no key


async def replay(polls: int = 10, interval_secs: float = 3.0) -> None:
    breaker = CircuitBreaker(BreakerConfig(), [log_event])
    for _ in range(polls):
        try:
            for metrics in await asyncio.to_thread(client.get_toxic_pairs):
                breaker.on_toxicity(metrics)
            for snapshot in await asyncio.to_thread(client.get_confluence_snapshots):
                breaker.on_confluence(snapshot)
        except RateLimitExceededException as exc:
            print(exc)
            print("Upgrade for 300-1,000 req/min and WebSocket streams: https://follow-sm.com/pricing")
            break
        await asyncio.sleep(interval_secs)
    await breaker.drain()
    print("Breaker evaluation latency:", breaker.latency.summary())


await replay()

## Enterprise: live tick-level breaker (60 seconds)

`/ws/v1/toxicity` and `/ws/v1/confluence` reject non-Enterprise keys with HTTP 403; the breaker then exits with an upgrade prompt.

In [ ]:
import getpass
import os

os.environ["FOLLOWSM_API_KEY"] = getpass.getpass("ENTERPRISE FollowSM API key: ").strip()
os.environ["SYMBOLS"] = "BTCUSDT,ETHUSDT,SOLUSDT"
!timeout -s INT 60 python -m circuit_breaker